# PGD $(x, P)$ trials of the clamped-clamped von Kármán beam

Separated deflection:
$$
w(x, P) = \sum_{m=1}^{M} w_m(x)\,\lambda_m(P), \qquad x \in [0, 1],\quad P \in [P_{\min}, P_{\max}].
$$

Exact solution on the branch $\psi_1$ (lowest energy), with $P_1 = 4\pi^2$:
$$
w(x, P) = c(P)\,\psi_1(x), \qquad \psi_1(x) = 1 - \cos 2\pi x, \qquad
c(P) = \begin{cases} 2\sqrt{P/P_1 - 1} & P > P_1 \\ 0 & P \le P_1 \end{cases}
$$

Energy at fixed $P$: $\Pi(P) = -\tfrac{1}{2}(P - P_1)^2$ above $P_1$, $0$ below. Integrated over the load range:
$$
J_{\text{exact}} = \int_{P_{\min}}^{P_{\max}} \Pi(P)\,\mathrm{d}P
= -\frac{\big(\max(P_{\max}, P_1) - P_1\big)^3 - \big(\max(P_{\min}, P_1) - P_1\big)^3}{6},
$$
i.e. $J_{\text{exact}} = -(P_{\max} - P_1)^3/6$ when $P_{\min} \le P_1$.

Run the setup cell once, then copy and edit the trial cell as you like.


In [8]:
import torch

import sys
sys.path.insert(0, "../..")
from main_pgd import main
from neurom.training import (
    Phase, RelativeChange, FixedIterations, MaxStages, RelativeGain, AnyOf,
)
from modes import symmetric_mode, antisymmetric_mode, exact_solution
from plotting import plot_loss_history, plot_monoms, plot_amplitude, plot_deformed_beam

torch.set_default_dtype(torch.float64)

P1 = 4 * torch.pi**2  # first critical load, clamped-clamped

# Physical constants (steel, 1 cm x 1 cm square section)
E = 210e9
b = 1e-2
A = b**2
I = b**4 / 12
L = 1.0


def exact_energy_integrated(P_range, P_critical=P1):
    """int_{P_min}^{P_max} -(P - P1)^2 / 2 dP over the buckled part (0 below P1)."""
    lo = max(P_range[0], P_critical)
    hi = max(P_range[1], P_critical)
    return -((hi - P_critical) ** 3 - (lo - P_critical) ** 3) / 6


def report(r, P_range):
    """Print a summary of a run against the exact branch c(P) psi1.

    Per mode: energy, gap to J_exact, gain over the previous mode, iterations
    per phase, stop reason of the last phase and weight of the mode
    ||w_m lambda_m|| / ||w||. Then the enrichment stop reason, the sign picked
    by the PGD and the errors on w. Norms are discrete L2 norms on the uniform
    sampling grid.

    Returns:
        torch.Tensor: the exact w on the grid, with the sign of the PGD.
    """
    history = r["history"]
    J_exact = exact_energy_integrated(P_range)
    w = r["w"]
    w_norm = torch.linalg.norm(w)

    print(f"J_exact = {J_exact:.6e}")
    previous = None
    for rec, wx, lp in zip(history.stages, r["x_monoms"], r["P_monoms"]):
        counts = {}
        for name in rec.phases:
            counts[name] = counts.get(name, 0) + 1
        iters = " + ".join(f"{name} {n}" for name, n in counts.items())
        gain = "" if previous is None else (
            f", gain = {previous - rec.energy:.3e} "
            f"({(previous - rec.energy) / max(abs(rec.energy), abs(previous)):.3e} rel)"
        )
        weight = torch.linalg.norm(torch.outer(wx, lp)) / w_norm
        print(
            f"mode {rec.stage}: J = {rec.energy:.6e}, "
            f"(J - J_exact)/|J_exact| = {(rec.energy - J_exact) / abs(J_exact):.3e}{gain}\n"
            f"    {iters} it, last phase {rec.stop_reason}, weight {weight:.3e}"
        )
        previous = rec.energy
    print(f"enrichment stopped: {history.stop_reason}")

    # Sign of the PGD solution, from the projection on the exact one
    w_ex = exact_solution(r["x"], r["P"])
    sign = 1.0 if (w * w_ex).sum() >= 0 else -1.0
    w_ex = sign * w_ex
    print(f"sign of c(P): {int(sign):+d}")

    err = (w - w_ex).abs()
    k = int(torch.argmax(err))
    i, j = divmod(k, err.shape[1])
    print(f"L2 error  ||w - w_exact|| / ||w_exact|| = {torch.linalg.norm(w - w_ex) / torch.linalg.norm(w_ex):.3e}")
    print(
        f"max error max|w - w_exact| / max|w_exact| = {err.max() / w_ex.abs().max():.3e}"
        f" at x = {r['x'][i]:.3f}, P = {r['P'][j]:.3f} (P/P1 = {r['P'][j] / P1:.3f})"
    )
    below = r["P"] <= P1
    if below.any():
        print(f"max|w| for P <= P1 (exact 0) = {w[:, below].abs().max():.3e}")
    above = ~below
    if above.any():
        mid = r["x"].shape[0] // 2  # x = 1/2
        err_mid = (w[mid, above] - w_ex[mid, above]).abs()
        j = int(torch.argmax(err_mid))
        print(
            f"amplitude for P > P1: max|w(1/2) - 2c| / max|2c| = "
            f"{err_mid.max() / w_ex[mid, above].abs().max():.3e}"
            f" at P/P1 = {r['P'][above][j] / P1:.3f}"
        )
    return w_ex


## Trial (template cell)

In [15]:
P_RANGE = (0.25 * P1, 3 * P1)

# Phases run on every mode, in order, each with its own stopping criterion:
# RelativeChange(tol, max_iter) or FixedIterations(n_iter). Remove a phase to skip it.
phases = [
    Phase(
        lambda p: torch.optim.Adam(p, lr=1e-2),
        RelativeChange(tol=1e-6, max_iter=5000),
        "adam",
    ),
    Phase(
        lambda p: torch.optim.LBFGS(p, lr=1.0, line_search_fn="strong_wolfe"),
        RelativeChange(tol=1e-6, max_iter=10),
        "lbfgs",
    ),
]

r = main(
    n_elements=10,
    n_P=100,
    P_range=P_RANGE,
    w_init=symmetric_mode(eps=1e-2, m=1),
    phases=phases,
    enrichment_criterion=AnyOf(RelativeGain(tol=1e-4), MaxStages(10)),  # number of modes
    # or stop when a mode barely lowers the energy, with a cap on the number of modes:
    # enrichment_criterion=MaxStages(3)
    E=E, A=A, I=I, length=L,
)
w_ex = report(r, P_RANGE)


* Training
mode 0: 400 iterations, converged, energy = -8.201235e+04
mode 1: 60 iterations, converged, energy = -8.201237e+04
J_exact = -8.203854e+04
mode 0: J = -8.201235e+04, (J - J_exact)/|J_exact| = 3.193e-04
    adam 397 + lbfgs 3 it, last phase converged, weight 1.000e+00
mode 1: J = -8.201237e+04, (J - J_exact)/|J_exact| = 3.190e-04, gain = 2.017e-02 (2.460e-07 rel)
    adam 57 + lbfgs 3 it, last phase converged, weight 1.246e-04
enrichment stopped: converged
sign of c(P): +1
L2 error  ||w - w_exact|| / ||w_exact|| = 1.206e-03
max error max|w - w_exact| / max|w_exact| = 6.602e-03 at x = 0.500, P = 39.182 (P/P1 = 0.992)
max|w| for P <= P1 (exact 0) = 3.735e-02
amplitude for P > P1: max|w(1/2) - 2c| / max|2c| = 2.589e-03 at P/P1 = 1.006


In [16]:
history = r["history"]
plot_loss_history(history.losses, [rec.stage for rec in history.stages for _ in rec.losses]).show()
plot_monoms(r["x"], r["x_monoms"], r["P"], r["P_monoms"]).show()

mid = r["x"].shape[0] // 2  # x = 1/2
fig = plot_amplitude(r["P"], r["w"][mid], P_critical=P1)
fig.add_scatter(x=r["P"].numpy(), y=w_ex[mid].numpy(), mode="lines", name="exact", line=dict(dash="dot"))
fig.show()

P_PLOT = 1.5 * P1
j = int(torch.argmin(torch.abs(r["P"] - P_PLOT)))
plot_deformed_beam(
    r["x_fine_hat"], r["w_fine_hat"][:, j], r["u_fine_hat"][:, j],
    x_nodes=r["x_hat"], w_nodes=r["w_hat"][:, j], u_nodes=r["u_hat"][:, j],
    amplification=5.0,
).show()